# Module 2 — Topic 2: Introduction to Pandas
## Live Demo Notebook

**Scenario:** You've received the full `orders.csv` export from the startup's database — 500 rows, 14 columns. In Topic 1 we extracted a single numeric column into NumPy. Now we load the whole table and start making sense of it as a structured dataset.

---

## Part 1 — Why Pandas? The Limits of NumPy

In [ ]:
import numpy as np

# NumPy handles a single column of numbers well
amounts = np.array([4500, 12000, 800, 33000, 7200])
print("Amounts:", amounts)
print("Mean:   ", np.mean(amounts))

# But real data is a table — mixed types, named columns
# Trying to mix strings and numbers in NumPy:
mixed = np.array(["ORD-001", "Ngozi Okafor", "Lagos", 4500])
print("\nMixed array:", mixed)
print("Dtype:", mixed.dtype)  # everything becomes a string!
print("Can we sum?", end=" ")
try:
    print(np.sum(mixed))
except TypeError as e:
    print(f"TypeError — {e}")

---
## Part 2 — Your First DataFrame

In [ ]:
import pandas as pd

print("Pandas version:", pd.__version__)

In [ ]:
# A Series — a single labelled column
revenue_by_state = pd.Series(
    [842000, 531000, 394000, 287000, 215000],
    index=["Lagos", "Abuja", "Rivers", "Kano", "Oyo"]
)

print(revenue_by_state)
print()
print("Lagos revenue: NGN", revenue_by_state["Lagos"])
print("Dtype:", revenue_by_state.dtype)
print("Index:", revenue_by_state.index.tolist())

In [ ]:
# A DataFrame — built from a dictionary (keys = column names)
sample_orders = pd.DataFrame({
    "order_id":    ["ORD-001", "ORD-002", "ORD-003", "ORD-004", "ORD-005"],
    "customer":    ["Ngozi Okafor", "Emeka Bello", "Fatima Musa", "Tunde Lawal", "Aisha Garba"],
    "state":       ["Lagos", "Abuja", "Kano", "Rivers", "Kaduna"],
    "category":    ["Electronics", "Fashion", "Food & Drinks", "Electronics", "Beauty & Health"],
    "amount":      [4500, 12000, 800, 33000, 5600],
    "delivered":   [True, True, False, True, False],
})

sample_orders

In [ ]:
# Key DataFrame attributes
print("Shape:  ", sample_orders.shape)          # (rows, columns)
print("Columns:", sample_orders.columns.tolist())
print("Dtypes:\n", sample_orders.dtypes)
print()

# Each column is a Series
print("Type of a column:", type(sample_orders["amount"]))

---
## Part 3 — Loading the Real Dataset

In [ ]:
# Load the full orders dataset
df = pd.read_csv("../../../data/orders.csv", encoding="utf-8-sig")

print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} columns")

In [ ]:
# Always start with head() — first 5 rows
df.head()

In [ ]:
# tail() — last 5 rows
# Look carefully — do these look different from the first rows?
df.tail()

---
## Part 4 — Understanding the Structure

In [ ]:
# info() — column names, dtypes, and missing value counts
df.info()

In [ ]:
# What we noticed from info():
# 1. phone has 492 non-null (8 missing)
# 2. total_amount has 494 non-null (6 missing)
# 3. unit_price is 'object' — it should be a number (has ₦ prefix)
# 4. order_date is 'object' — it should be a datetime

# Let's confirm the unit_price issue
print("unit_price samples:")
print(df["unit_price"].head(10).tolist())

In [ ]:
# describe() — statistical summary of numeric columns
df.describe()

In [ ]:
# Key observations from describe():
mean_val   = df["total_amount"].mean()
median_val = df["total_amount"].median()
max_val    = df["total_amount"].max()

print(f"Mean order value:   NGN {mean_val:,.0f}")
print(f"Median order value: NGN {median_val:,.0f}")
print(f"Max order value:    NGN {max_val:,.0f}")
print()
print(f"Gap (mean - median): NGN {mean_val - median_val:,.0f}")
print("→ Outliers are pulling the mean up")

---
## Part 5 — Selecting Columns

In [ ]:
# Single column → Series
states = df["state"]
print("Type:", type(states))
print(states.head(8))

In [ ]:
# Multiple columns → DataFrame (double brackets)
subset = df[["customer_name", "state", "product_category", "total_amount"]]
print("Type:", type(subset))
subset.head(8)

In [ ]:
# Useful Series operations on a column
print("Unique categories:")
print(df["product_category"].unique())
print()

print("Orders per category:")
print(df["product_category"].value_counts())
print()

print("Orders per delivery status:")
print(df["delivery_status"].value_counts())

---
## Part 6 — Filtering Rows

In [ ]:
# All orders from Abuja
# Note: state column has inconsistent casing — we'll see that here
abuja_orders = df[df["state"] == "Abuja"]
print(f"Orders with state == 'Abuja': {len(abuja_orders)}")

# Why might this be fewer than expected?
print("\nAll unique state values (showing the inconsistency):")
print(sorted(df["state"].unique()))

In [ ]:
# High-value orders — above NGN 30,000
high_value = df[df["total_amount"] > 30000]
print(f"High-value orders (> NGN 30,000): {len(high_value)}")
high_value[["order_id", "customer_name", "product_category", "total_amount"]].head(10)

In [ ]:
# Multiple conditions — Electronics orders above NGN 10,000
electronics_high = df[
    (df["product_category"] == "Electronics") &
    (df["total_amount"] > 10000)
]
print(f"Electronics orders > NGN 10,000: {len(electronics_high)}")
electronics_high[["order_id", "state", "product_name", "total_amount"]].head(8)

In [ ]:
# .isin() — match any value in a list
# Delivered or Shipped orders
active_orders = df[df["delivery_status"].isin(["Delivered", "Shipped"])]
print(f"Active orders (Delivered or Shipped): {len(active_orders)}")
print(active_orders["delivery_status"].value_counts())

---
## Part 7 — `loc` and `iloc`

In [ ]:
# iloc — integer position (like NumPy)
print("First row (iloc[0]):")
print(df.iloc[0])
print()

print("First 3 rows, first 4 columns (iloc[0:3, 0:4]):")
print(df.iloc[0:3, 0:4])

In [ ]:
# loc — label-based selection
print("Row 0, customer_name column:")
print(df.loc[0, "customer_name"])
print()

print("Rows 0–4, order_id to state:")
print(df.loc[0:4, "order_id":"state"])

In [ ]:
# loc with a condition — filter rows AND select specific columns in one step
high_value_customers = df.loc[
    df["total_amount"] > 30000,
    ["customer_name", "state", "product_category", "total_amount"]
]

print(f"High-value customers ({len(high_value_customers)} orders):")
high_value_customers.head(10)

---
## Part 8 — Saving Results

In [ ]:
# Save high-value orders to a new CSV
high_value_customers.to_csv(
    "../../../data/high_value_orders.csv",
    index=False,
    encoding="utf-8-sig"
)

# Verify it saved correctly by reloading
verify = pd.read_csv("../../../data/high_value_orders.csv", encoding="utf-8-sig")
print(f"Saved and reloaded: {verify.shape[0]} rows x {verify.shape[1]} columns")
verify.head()

---
## Part 9 — Putting It All Together

**Business question:** *"Give me a summary of Electronics orders — how many, total revenue, and average order value — broken down by payment method."*

In [ ]:
# Step 1: Filter to Electronics only
electronics = df[df["product_category"] == "Electronics"].copy()
print(f"Electronics orders: {len(electronics)}")

# Step 2: Drop rows with missing total_amount (can't calculate revenue without it)
electronics = electronics.dropna(subset=["total_amount"])
print(f"After dropping missing amounts: {len(electronics)}")

In [ ]:
# Step 3: Summary by payment method
payment_summary = electronics.groupby("payment_method")["total_amount"].agg(
    order_count="count",
    total_revenue="sum",
    avg_order="mean"
).round(0)

payment_summary

In [ ]:
# Step 4: Format and present
print("===== Electronics Orders by Payment Method =====")
for method, row in payment_summary.iterrows():
    print(f"  {method:<20} | Orders: {int(row['order_count']):>3} | "
          f"Revenue: NGN {row['total_revenue']:>10,.0f} | "
          f"Avg: NGN {row['avg_order']:>8,.0f}")

---
## Summary

In this demo we:
- Saw why NumPy can't handle mixed-type tables
- Created a Series and a DataFrame from scratch
- Loaded `orders.csv` with `pd.read_csv()`
- Inspected the dataset with `head()`, `tail()`, `info()`, and `describe()`
- Spotted real problems: missing values, wrong dtypes, inconsistent casing
- Selected columns (single and multiple) and explored them with `value_counts()`
- Filtered rows with single and multiple conditions and `.isin()`
- Used `loc` and `iloc` for precise selection
- Saved results to a CSV
- Built a complete business summary combining filtering, groupby, and aggregation

**Next — Topic 3:** Our dataset has real problems — missing values, a `unit_price` column stored as a string, inconsistent state casing, and duplicate rows. Topic 3 is dedicated to fixing all of them.